## Human Posture Detection

[<img src ="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" align="left">](https://colab.research.google.com/github/spmallick/learnopencv/blob/master/Posture-analysis-system-using-MediaPipe-Pose/human_posture_analysis.ipynb)

Poor posture is a modern-day epidemic. Research has shown that children as young as 10 years of age are demonstrating spinal degeneration on x-ray. Certain postures, like forward head posture, have been linked to migraines, high blood pressure, and decreased lung capacity. In this notebook, you will learn to create an application to detect posture using mediapipe. We are going to use side view samples to perform analysis and draw conclusion. Practical application would require an webcam pointed at the side view of the person.

This notebook supports both **video and image processing modes**, allowing you to analyze posture in real-time video streams or static images using the same MediaPipe Pose detection algorithms.

### Workflow
 - Find point of interests (landmarks) to check angle.
 - Perform analysis on standard sample images.
 - Find threshold range for good and bad posture.
 - Apply on video/webcam input or single images.

### Goal
To detect neck inclination and torso inclination as shown below.
<br>
<img src = "https://learnopencv.com/wp-content/uploads/2022/03/mp-pose-03-posture-sitting-scaled.jpg" align='center'>

In [1]:
if 'google.colab' in str(get_ipython()):
    !pip install opencv-python
    !pip install mediapipe
    !wget https://raw.githubusercontent.com/spmallick/learnopencv/blob/master/Posture-analysis-system-using-MediaPipe-Pose/input.mp4
else:
    pass

## Import Libraries

In [ ]:
import cv2
import time
import math as m
import mediapipe as mp
import os

In [ ]:
import logging
import urllib.request

logger = logging.getLogger(__name__)

# --- MediaPipe Compatibility Shim ---
def _ensure_pose_model() -> str:
    """Download the pose landmarker model if it is not already present."""
    import tempfile
    model_dir = os.path.join(tempfile.gettempdir(), "mediapipe_models")
    model_path = os.path.join(model_dir, "pose_landmarker_lite.task")
    model_url = (
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    )
    if not os.path.exists(model_path):
        os.makedirs(model_dir, exist_ok=True)
        logger.info("Downloading MediaPipe model to %s", model_path)
        urllib.request.urlretrieve(model_url, model_path)
    return model_path

class _FakeLandmark:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

class _FakePoseLandmarks:
    def __init__(self, landmark):
        self.landmark = landmark

class _FakeResults:
    def __init__(self, pose_landmarks):
        self.pose_landmarks = pose_landmarks

class _PoseLandmarkIndex:
    LEFT_EAR = 7
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    LEFT_HIP = 23

class _FakeMpPose:
    PoseLandmark = _PoseLandmarkIndex
    POSE_CONNECTIONS = frozenset([
        (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
        (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21),
        (17, 19), (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (11, 23), (12, 24), (23, 24), (23, 25), (24, 26), (25, 27), (26, 28),
        (27, 29), (28, 30), (29, 31), (30, 32), (27, 31), (28, 32)
    ])
    @staticmethod
    def Pose(**kwargs):
        return _TasksPoseShim(**kwargs)

class _TasksPoseShim:
    """Shim so code written for mp.solutions.pose keeps working on latest MediaPipe."""
    def __init__(self, static_image_mode=False, **kwargs):
        if not hasattr(mp, "tasks"):
            raise RuntimeError(
                "MediaPipe Tasks API is not available in this installation. "
                "Please install a supported mediapipe package."
            )
        model_path = _ensure_pose_model()
        base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
        
        running_mode = (
            mp.tasks.vision.RunningMode.IMAGE 
            if static_image_mode 
            else mp.tasks.vision.RunningMode.VIDEO
        )
        
        options = mp.tasks.vision.PoseLandmarkerOptions(
            base_options=base_options,
            running_mode=running_mode,
            num_poses=1,
        )
        self._landmarker = mp.tasks.vision.PoseLandmarker.create_from_options(options)

    def process(self, rgb_image):
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)
        try:
            result = self._landmarker.detect(mp_image)
        except ValueError:
            if not hasattr(self, '_timestamp_ms'):
                self._timestamp_ms = 0
            self._timestamp_ms += 33 # approx 30 fps
            result = self._landmarker.detect_for_video(mp_image, self._timestamp_ms)

        if not result.pose_landmarks:
            return _FakeResults(None)
        landmarks = [_FakeLandmark(lm.x, lm.y) for lm in result.pose_landmarks[0]]
        return _FakeResults(_FakePoseLandmarks(landmarks))

    def close(self):
        self._landmarker.close()

# Provide mp_pose compatible module
if hasattr(mp, "solutions"):
    mp_pose_module = mp.solutions.pose
else:
    logger.info("Top-level mediapipe.solutions not available; using tasks compatibility shim.")
    mp_pose_module = _FakeMpPose

# Initialize mediapipe selfie segmentation / pose class
mp_pose = mp_pose_module
if hasattr(mp, "solutions") and hasattr(mp.solutions, "holistic"):
    mp_holistic = mp.solutions.holistic
# --- End Shim ---

## Function to calculate offset distance
The setup requires the person to be in proper side view. **`findDistance`** function helps us determine offset distance between two poinst. It can be the hip points, the eyes or shoulder. As these points are always more or less symmetric about the central axis. With this, we are going to incorporate camera alignment assistance in the script. Distnace is calculated using the distance formula.


\begin{align}
distance =  \sqrt{(x2 - x1)^2+(y2 - y1)^2}
\end{align}

In [3]:
def find_distance(x1, y1, x2, y2):
    """Return Euclidean distance between two points."""
    
    return m.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

## Function to calculate angle subtended by the line of interest to y-axis
This is the primary deterministic factor for the posture. Using the angle subtended by the neck line and the torso line to y-axis. The neck line connects the shoulder and the eye, here we take shoulder as the pivotal point. Similarly the torso line connects hip and the shoulder, where hip is considered pivotal point.
<br>

<img src="https://learnopencv.com/wp-content/uploads/2022/03/mp-pose-05-neckline-inclination.jpg" alt="MediaPipe pose neck inclination" align="middle" width="500" height="600">

<br>

Taking neck line as an example, here points are $P_1(x_1, y_1)$(shoulder), $P_2(x_2, y_2)$ (eye) and $P_3(x_3, y_3)$ (any point on vertical axis passing through $P_1$).
<br>
Hence, for $P_3$ x-coordinate is same as to that of $P_1$ and since $y_3$ is valid for all y, let's take $y_3 = 0$ for simplicity. <br>To find the inner angle of three points, we take vector approach. Angle between two vectors $\vec{P_{12}}$ and $\vec{P_{13}}$ is given by,
\begin{align}
\theta = \arccos (\frac{\vec{P_{12}}.\vec{P_{13}}}{|\vec{P_{12}}|.|\vec{P_{13}}|})
\end{align}
Solving for $\theta$ we get,
\begin{align}
\theta = \arccos (\frac{y_1^2 - y_1.y_2}{y_1\sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}})
\end{align}

In [4]:
def find_angle(x1, y1, x2, y2):
    """Return the inclination angle between two points in degrees."""
    
    denominator = m.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2) * y1
    if denominator == 0:
        return 0
    theta = m.acos((y2 - y1) * (-y1) / denominator)
    return int(180 / m.pi) * theta

## Function to send alert
Use this function to send alerts when bad posture is detected. Feel free to get creative and customize as per your convenience. You can use telegram notifiction alert system, [chek out Telegram bot automation here](https://core.telegram.org/bots). Or you can take it up a notch by creating an android app.

In [5]:
def send_warning(duration):
    """Send an alert when bad posture persists for *duration* seconds."""
    
    print(f"Warning: Bad posture detected for {duration:.2f} seconds")

## Constants and Initializations

In [6]:
# Initilize frame counters.
good_frames = 0
bad_frames = 0

# Font type.
font = cv2.FONT_HERSHEY_SIMPLEX

# Colors.
blue = (255, 127, 0)
red = (50, 50, 255)
green = (127, 255, 0)
dark_blue = (127, 20, 0)
light_green = (127, 233, 100)
yellow = (0, 255, 255)
pink = (255, 0, 255)

In [ ]:
# Initialize mediapipe pose class.
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()

# Initialize video capture object.
# For webcam input replace file name with 0.
file_name = '../data/video/input2.mp4'
cap = cv2.VideoCapture(file_name)

# Meta.
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Initialize video writer.
video_output = cv2.VideoWriter('../data/video/output2.mp4', fourcc, fps, frame_size)

## Processing

Processing the image using mediapipe is fairly simple, after setting minimum detection confidence and minimum tracking confidence, we need to pass the RGB image to `pose.process()` method. The documentation can be found in this [**link**](https://google.github.io/mediapipe/solutions/pose). With the acquired result, we find the coordinates of specific landmarks. Then we find the angles suntended by the line of interset to y-axis and draw conclusion on the basis of results obtained from analysis script. The code is self explanatory.

In [ ]:
print('Processing..')
while cap.isOpened():
    # Capture frames.
    success, image = cap.read()
    if not success:
        print("Null.Frames")
        break
    # Get fps.
    fps = cap.get(cv2.CAP_PROP_FPS)
    # Get height and width.
    h, w = image.shape[:2]

    # Convert the BGR image to RGB.
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Process the image.
    keypoints = pose.process(image)

    # Convert the image back to BGR.
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    # Use lm and lmPose as representative of the following methods.
    lm = keypoints.pose_landmarks
    lmPose = mp_pose.PoseLandmark

    # Acquire the landmark coordinates.
    # Once aligned properly, left or right should not be a concern.
    # Left shoulder.
    l_shldr_x = int(lm.landmark[lmPose.LEFT_SHOULDER].x * w)
    l_shldr_y = int(lm.landmark[lmPose.LEFT_SHOULDER].y * h)
    # Right shoulder
    r_shldr_x = int(lm.landmark[lmPose.RIGHT_SHOULDER].x * w)
    r_shldr_y = int(lm.landmark[lmPose.RIGHT_SHOULDER].y * h)
    # Left ear.
    l_ear_x = int(lm.landmark[lmPose.LEFT_EAR].x * w)
    l_ear_y = int(lm.landmark[lmPose.LEFT_EAR].y * h)
    # Left hip.
    l_hip_x = int(lm.landmark[lmPose.LEFT_HIP].x * w)
    l_hip_y = int(lm.landmark[lmPose.LEFT_HIP].y * h)

    # Calculate distance between left shoulder and right shoulder points.
    offset = find_distance(l_shldr_x, l_shldr_y, r_shldr_x, r_shldr_y)

    # Assist to align the camera to point at the side view of the person.
    # Offset threshold 30 is based on results obtained from analysis over 100 samples.
    if offset < 100:
        cv2.putText(image, str(int(offset)) + ' Aligned', (w - 150, 30), font, 0.9, green, 2)
    else:
        cv2.putText(image, str(int(offset)) + ' Not Aligned', (w - 150, 30), font, 0.9, red, 2)

    # Calculate angles.
    neck_inclination = find_angle(l_shldr_x, l_shldr_y, l_ear_x, l_ear_y)
    torso_inclination = find_angle(l_hip_x, l_hip_y, l_shldr_x, l_shldr_y)

    # Draw landmarks.
    cv2.circle(image, (l_shldr_x, l_shldr_y), 7, yellow, -1)
    cv2.circle(image, (l_ear_x, l_ear_y), 7, yellow, -1)

    # Let's take y - coordinate of P3 100px above x1,  for display elegance.
    # Although we are taking y = 0 while calculating angle between P1,P2,P3.
    cv2.circle(image, (l_shldr_x, l_shldr_y - 100), 7, yellow, -1)
    cv2.circle(image, (r_shldr_x, r_shldr_y), 7, pink, -1)
    cv2.circle(image, (l_hip_x, l_hip_y), 7, yellow, -1)

    # Similarly, here we are taking y - coordinate 100px above x1. Note that
    # you can take any value for y, not necessarily 100 or 200 pixels.
    cv2.circle(image, (l_hip_x, l_hip_y - 100), 7, yellow, -1)

    # Put text, Posture and angle inclination.
    # Text string for display.
    angle_text_string = 'Neck: ' + str(int(neck_inclination)) + '  Torso: ' + str(int(torso_inclination))

    # Determine whether good posture or bad posture.
    # The threshold angles have been set based on intuition.
    if neck_inclination < 40 and torso_inclination < 10:
        bad_frames = 0
        good_frames += 1

        cv2.putText(image, angle_text_string, (10, 30), font, 0.9, light_green, 2)
        cv2.putText(image, str(int(neck_inclination)), (l_shldr_x + 10, l_shldr_y), font, 0.9, light_green, 2)
        cv2.putText(image, str(int(torso_inclination)), (l_hip_x + 10, l_hip_y), font, 0.9, light_green, 2)

        # Join landmarks.
        cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), green, 4)
        cv2.line(image, (l_shldr_x, l_shldr_y), (l_shldr_x, l_shldr_y - 100), green, 4)
        cv2.line(image, (l_hip_x, l_hip_y), (l_shldr_x, l_shldr_y), green, 4)
        cv2.line(image, (l_hip_x, l_hip_y), (l_hip_x, l_hip_y - 100), green, 4)

    else:
        good_frames = 0
        bad_frames += 1

        cv2.putText(image, angle_text_string, (10, 30), font, 0.9, red, 2)
        cv2.putText(image, str(int(neck_inclination)), (l_shldr_x + 10, l_shldr_y), font, 0.9, red, 2)
        cv2.putText(image, str(int(torso_inclination)), (l_hip_x + 10, l_hip_y), font, 0.9, red, 2)

        # Join landmarks.
        cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), red, 4)
        cv2.line(image, (l_shldr_x, l_shldr_y), (l_shldr_x, l_shldr_y - 100), red, 4)
        cv2.line(image, (l_hip_x, l_hip_y), (l_shldr_x, l_shldr_y), red, 4)
        cv2.line(image, (l_hip_x, l_hip_y), (l_hip_x, l_hip_y - 100), red, 4)

    # Calculate the time of remaining in a particular posture.
    good_time = (1 / fps) * good_frames
    bad_time =  (1 / fps) * bad_frames

    # Pose time.
    if good_time > 0:
        time_string_good = 'Good Posture Time : ' + str(round(good_time, 1)) + 's'
        cv2.putText(image, time_string_good, (10, h - 20), font, 0.9, green, 2)
    else:
        time_string_bad = 'Bad Posture Time : ' + str(round(bad_time, 1)) + 's'
        cv2.putText(image, time_string_bad, (10, h - 20), font, 0.9, red, 2)

    # If you stay in bad posture for more than 3 minutes (180s) send an alert.
    if bad_time > 180:
        send_warning(bad_time)
    # Write frames.
    video_output.write(image)
print('Finished.')
cap.release()
video_output.release()

## Image Processing Mode

In [13]:
def process_image(input_path, output_path, display=True):
    """Process *input_path* image and write annotated result to *output_path*."""

    # Setting up MediaPipe pose detection for static images
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=True)
    
    # Loading image
    image = cv2.imread(input_path)
    if image is None:
        raise FileNotFoundError(f"Unable to load image: {input_path}")

    frame_height, frame_width = image.shape[:2]
    
    # Font type and colors (using existing notebook variables)
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    try:
        # Converting to RGB for MediaPipe processing
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        keypoints = pose.process(rgb_image)
        image = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)

        landmarks = keypoints.pose_landmarks
        if landmarks is None:
            print("No pose landmarks detected in the image.")
            # Adding a message to the image
            cv2.putText(image, "No pose detected", (10, 30), font, 0.9, red, 2)
        else:
            # Processing pose landmarks similar to video processing
            lm_pose = mp_pose.PoseLandmark
            l_shldr_x = int(landmarks.landmark[lm_pose.LEFT_SHOULDER].x * frame_width)
            l_shldr_y = int(landmarks.landmark[lm_pose.LEFT_SHOULDER].y * frame_height)
            r_shldr_x = int(landmarks.landmark[lm_pose.RIGHT_SHOULDER].x * frame_width)
            r_shldr_y = int(landmarks.landmark[lm_pose.RIGHT_SHOULDER].y * frame_height)
            l_ear_x = int(landmarks.landmark[lm_pose.LEFT_EAR].x * frame_width)
            l_ear_y = int(landmarks.landmark[lm_pose.LEFT_EAR].y * frame_height)
            l_hip_x = int(landmarks.landmark[lm_pose.LEFT_HIP].x * frame_width)
            l_hip_y = int(landmarks.landmark[lm_pose.LEFT_HIP].y * frame_height)

            # Calculating alignment and posture metrics
            offset = find_distance(l_shldr_x, l_shldr_y, r_shldr_x, r_shldr_y)
            if offset < 100:
                cv2.putText(image, str(int(offset)) + ' Aligned', (frame_width - 150, 30), font, 0.9, green, 2)
            else:
                cv2.putText(image, str(int(offset)) + ' Not Aligned', (frame_width - 150, 30), font, 0.9, red, 2)

            neck_inclination = find_angle(l_shldr_x, l_shldr_y, l_ear_x, l_ear_y)
            torso_inclination = find_angle(l_hip_x, l_hip_y, l_shldr_x, l_shldr_y)

            # Drawing key landmarks
            cv2.circle(image, (l_shldr_x, l_shldr_y), 7, yellow, -1)
            cv2.circle(image, (l_ear_x, l_ear_y), 7, yellow, -1)
            cv2.circle(image, (l_shldr_x, l_shldr_y - 100), 7, yellow, -1)
            cv2.circle(image, (r_shldr_x, r_shldr_y), 7, pink, -1)
            cv2.circle(image, (l_hip_x, l_hip_y), 7, yellow, -1)
            cv2.circle(image, (l_hip_x, l_hip_y - 100), 7, yellow, -1)

            angle_text = f"Neck: {int(neck_inclination)}  Torso: {int(torso_inclination)}"

            # Determining posture quality and applying appropriate annotations
            if neck_inclination < 40 and torso_inclination < 10:
                cv2.putText(image, angle_text, (10, 30), font, 0.9, light_green, 2)
                cv2.putText(image, str(int(neck_inclination)), (l_shldr_x + 10, l_shldr_y), font, 0.9, light_green, 2)
                cv2.putText(image, str(int(torso_inclination)), (l_hip_x + 10, l_hip_y), font, 0.9, light_green, 2)
                cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), green, 4)
                cv2.line(image, (l_shldr_x, l_shldr_y), (l_shldr_x, l_shldr_y - 100), green, 4)
                cv2.line(image, (l_hip_x, l_hip_y), (l_shldr_x, l_shldr_y), green, 4)
                cv2.line(image, (l_hip_x, l_hip_y), (l_hip_x, l_hip_y - 100), green, 4)
                cv2.putText(image, "Good Posture", (10, frame_height - 20), font, 0.9, green, 2)
            else:
                cv2.putText(image, angle_text, (10, 30), font, 0.9, red, 2)
                cv2.putText(image, str(int(neck_inclination)), (l_shldr_x + 10, l_shldr_y), font, 0.9, red, 2)
                cv2.putText(image, str(int(torso_inclination)), (l_hip_x + 10, l_hip_y), font, 0.9, red, 2)
                cv2.line(image, (l_shldr_x, l_shldr_y), (l_ear_x, l_ear_y), red, 4)
                cv2.line(image, (l_shldr_x, l_shldr_y), (l_shldr_x, l_shldr_y - 100), red, 4)
                cv2.line(image, (l_hip_x, l_hip_y), (l_shldr_x, l_shldr_y), red, 4)
                cv2.line(image, (l_hip_x, l_hip_y), (l_hip_x, l_hip_y - 100), red, 4)
                cv2.putText(image, "Poor Posture", (10, frame_height - 20), font, 0.9, red, 2)

        # Saving the annotated image
        cv2.imwrite(output_path, image)
        print(f"Annotated image saved to: {output_path}")

        # Displaying the image if requested
        if display:
            cv2.imshow("MediaPipe Pose - Image", image)
            cv2.waitKey(0)
            cv2.destroyAllWindows()

    finally:
        pose.close()

## Processing Sample Image

In [ ]:
# Example usage for image processing
# Note: Make sure you have a sample image in the specified path
input_image_path = '../data/images/input.png'
output_image_path = '../data/images/output.png'

try:
    process_image(input_image_path, output_image_path, display=True)
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure you have a sample image at the specified path.")
except Exception as e:
    print(f"An error occurred: {e}")

print("Image processing function is ready to use!")
print("To process an image, uncomment the lines above and ensure you have an image at '../data/images/input.png'")